In [ ]:
!pip install ipeadatapy
import ipeadatapy
import pandas as pd


def gerar_tabela_populacao():
    """Gera tabela de população total dos municípios selecionados."""

    pop = ipeadatapy.timeseries("POPTOT").reset_index(drop=True)
    pop = pop.reset_index()
    pop.columns = (
        pop.columns.str.lower()
        .str.replace("(", "")
        .str.replace(")", "")
        .str.replace(" ", "_")
    )
    pop = pop[["tercodigo", "year", "value_habitante"]]

    munic_selecionados = {
        "3548708": "São Bernardo do Campo",
        "3534401": "Osasco",
        "3552205": "Sorocaba",
        "3543402": "Ribeirão Preto",
        "3547809": "Santo André",
        "3549904": "São José dos Campos",
    }
    pop_munic_selecionados = pop.loc[
        pop["tercodigo"].isin(list(munic_selecionados.keys()))
        & (pop["year"] >= 1970)
    ].copy()
    pop_munic_selecionados["municipio"] = pop_munic_selecionados["tercodigo"].map(
        munic_selecionados
    )

    return pop_munic_selecionados


def calcular_densidade_demografica(pop_munic_selecionados):
    area = ipeadatapy.timeseries("AREA").reset_index(drop=True)

    area.columns = (
        area.columns.str.lower()
        .str.replace("(", "")
        .str.replace(")", "")
        .str.replace(" ", "_")
    )

    area = area[["tercodigo", "year", "value_km2"]].drop_duplicates()

    pop_munic_selecionados = pop_munic_selecionados.merge(
        area, on=["tercodigo", "year"], how="left"
    )

    pop_munic_selecionados["densidade_demografica"] = (
        pop_munic_selecionados["value_habitante"] / pop_munic_selecionados["value_km2"]
    )
    return pop_munic_selecionados


pop_munic_selecionados = calcular_densidade_demografica(gerar_tabela_populacao())

pop_munic_selecionados.to_csv(
    "/lakehouse/default/Files/gold_populacao_densidade/densidade_pop_munic_selecionados.csv",sep=";",
    encoding="latin1",
    index=False
)

StatementMeta(, 33e2c6a2-31f4-452d-a9e5-1c314fee01b5, 4, Finished, Available, Finished)